# real glasses

the colour notebook (10) showed that colour depends on the glass, not on the lens shape: three very different shapes all had about the same chromatic aberration (−0.78 mm). so the next question is which glass to make the lens from.

## the idea

**what a glass catalog lists.** glass makers (the biggest is SCHOTT) sell hundreds of optical glasses. each one is described by two headline numbers:

- **nd**: the refractive index for yellow-green light (587.6 nm). how strongly the glass bends light.
- **V**, the abbe number: how little the glass spreads colours apart. a big V means little colour.

| glass | nd | V | in plain words |
|---|---|---|---|
| N-FK51A | 1.48656 | 84.47 | weak bending, very little colour |
| N-BK7 | 1.51680 | 64.17 | the everyday standard glass |
| N-K5 | 1.52249 | 59.48 | similar to N-BK7 |
| N-SK16 | 1.62041 | 60.32 | strong bending, little colour |
| N-F2 | 1.62005 | 36.43 | strong bending, lots of colour |
| N-LAK9 | 1.69100 | 54.71 | stronger bending, little colour |
| N-SF11 | 1.78472 | 25.68 | strongest bending, the most colour |

(glasses get number codes from these two values: N-BK7 is "517642", nd 1.517 and V 64.2.)

**blue and red from nd and V.** the abbe number is defined as V = (nd − 1) / (n_blue − n_red). turned around, the gap between blue and red is

spread = (nd − 1) / V

and we split it evenly either side of nd: n_blue = nd + spread / 2, n_red = nd − spread / 2. real glass doesn't spread exactly evenly (blue sits a bit further from the middle than red), but the chromatic detector only uses the difference between blue and red, and this gets the difference exactly right.

**the same lens in any glass.** a surface bends light in proportion to (n − 1) / R: how much stronger the glass is than air, divided by how curved it is. so stronger glass needs gentler curves to do the same job. to compare glasses fairly, every lens keeps the best-form shape and f = 50 mm: the front radius is scaled by (n − 1) / 0.5 (so n = 1.5 gives back R1 = 29.5 mm), and R2_for sets the back radius for f = 50 mm in that glass.

**diagnose with colour.** diagnose now takes an optional glass. given one, it adds the chromatic detector as a fourth problem; without one, it works exactly as before.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("..")
from lens.raytrace import cross
from lens.design import R2_for
from lens.detect import spherical, field_curvature, coma, chromatic, diagnose
from lens.glass import catalog, colours

bk7 = catalog[1]
print(bk7)
print(colours(bk7))

{'name': 'N-BK7', 'nd': 1.5168, 'V': 64.17}
(1.5208268038023998, 1.5127731961976)


**the catalog.** each glass is a dictionary, and the catalog is a list of them. colours() turns a glass's nd and V into an index for blue and red: for N-BK7 the spread is 0.5168 / 64.17 = 0.00805, so blue is 1.52083 and red is 1.51277.

In [2]:
def lens_for(glass, t = 6.0):
    n = glass["nd"]
    R1 = 29.5 * (n - 1) / 0.5
    R2 = R2_for(R1, t, 50, n)
    return [(0, R1, n), (t, R2, 1.0)]

print(lens_for(catalog[1]))
print(lens_for(catalog[6]))

[(0, 30.491199999999996, 1.5168), (6.0, -158.03831223628688, 1.0)]
[(0, 46.298480000000005, 1.78472), (6.0, -242.5575105014917, 1.0)]


**one lens shape in any glass.** in N-BK7 the best-form lens has R1 = 30.5 mm and R2 = −158.0 mm. in N-SF11, the strongest-bending glass, it only needs R1 = 46.3 mm and R2 = −242.6 mm: much gentler curves for the same 50 mm focal length.

In [3]:
diagnose(lens_for(bk7), glass = bk7)

spherical:        -0.500 mm along the axis  → about 0.050 mm of blur
field curvature:  -2.550 mm along the axis  → about 0.255 mm of blur
coma:             +0.0210 mm on the sensor   → about 0.021 mm of blur
chromatic:        -0.756 mm along the axis  → about 0.076 mm of blur
main problem: field curvature
the edges of the image focus closer to the lens than the center, so a flat sensor can't catch both.
try: moving the aperture stop, bending the lens, or a compromise sensor position.


'field curvature'

**diagnose with colour.** the best-form lens in N-BK7, at f/5 with a 10° edge: field curvature causes about 0.255 mm of blur, chromatic 0.076 mm, spherical 0.050 mm and coma 0.021 mm. colour is the second-biggest problem, ahead of spherical aberration, which the best-form shape was optimized to reduce.

In [4]:
print("glass     spherical  field curv.  coma    chromatic   (mm of blur at f/5)")
for glass in catalog:
    lens = lens_for(glass)
    n_blue, n_red = colours(glass)
    sa = abs(spherical(lens, 5.0)) * 0.1
    fc = abs(field_curvature(lens)) * 0.1
    cm = abs(coma(lens))
    ch = abs(chromatic(lens, n_blue, n_red)) * 0.1
    print(f"{glass['name']:8s}  {sa:.3f}      {fc:.3f}        {cm:.3f}   {ch:.3f}")

glass     spherical  field curv.  coma    chromatic   (mm of blur at f/5)
N-FK51A   0.054      0.255        0.011   0.057
N-BK7     0.050      0.255        0.021   0.076
N-K5      0.049      0.250        0.023   0.082
N-SK16    0.040      0.250        0.047   0.080
N-F2      0.040      0.250        0.047   0.133
N-LAK9    0.036      0.255        0.060   0.089


N-SF11    0.033      0.250        0.073   0.189


**the whole catalog**, each glass making the same best-form shape with f = 50 mm:

| glass | V | spherical | field curvature | coma | chromatic |
|---|---|---|---|---|---|
| N-FK51A | 84.5 | 0.054 | 0.255 | 0.011 | **0.057** |
| N-BK7 | 64.2 | 0.050 | 0.255 | 0.021 | 0.076 |
| N-K5 | 59.5 | 0.049 | 0.250 | 0.023 | 0.082 |
| N-SK16 | 60.3 | 0.040 | 0.250 | 0.047 | 0.080 |
| N-F2 | 36.4 | 0.040 | 0.250 | 0.047 | 0.133 |
| N-LAK9 | 54.7 | 0.036 | 0.255 | 0.060 | 0.089 |
| N-SF11 | 25.7 | **0.033** | 0.250 | 0.073 | 0.189 |

(mm of blur at f/5, 10° edge)

- **colour follows V.** N-FK51A (the biggest V) has the least colour, N-SF11 (the smallest V) has 3.3x more. the chromatic blur is close to (f / V) × 0.1 every time.
- **spherical aberration falls as nd rises.** stronger glass needs gentler curves, and gentler curves bend the edge rays less violently: N-SF11 has 40% less spherical aberration than N-FK51A.
- **usually you can't have both.** high-index glasses tend to spread colour more (N-F2 and N-SF11). N-SK16 and N-LAK9 break that rule, with strong bending and little colour.
- **coma rises with nd here**, but that's because the shape was copied rather than re-optimized for each glass. the best bending depends on n.
- **field curvature doesn't move** (0.250 to 0.255 mm for every glass). changing the glass doesn't touch the biggest problem; that still needs the stop and the shape (notebook 08).

In [5]:
sf11 = catalog[6]
n = sf11["nd"]
R1 = 100 * (n - 1) / 0.5
flat = [(0, R1, n), (6, R2_for(R1, 6, 50, n), 1.0)]
diagnose(flat, z_stop = -15, glass = sf11)

spherical:        -0.877 mm along the axis  → about 0.088 mm of blur
field curvature:  -0.850 mm along the axis  → about 0.085 mm of blur
coma:             +0.0649 mm on the sensor   → about 0.065 mm of blur
chromatic:        -1.920 mm along the axis  → about 0.192 mm of blur
main problem: chromatic
blue light focuses closer to the lens than red, so white light smears into coloured fringes.
try: a glass with a bigger abbe number V (it spreads colours less).


'chromatic'

**making colour the main problem.** the flat-field shape from notebook 08 (front radius scaled the same way, stop 15 mm in front) made in N-SF11. flattening the field removed the biggest problem, and N-SF11's strong colour spread takes over: chromatic aberration causes about 0.192 mm of blur, more than twice anything else.

In [6]:
diagnose(flat, a = 2.5, z_stop = -15, glass = sf11)

spherical:        -0.218 mm along the axis  → about 0.011 mm of blur
field curvature:  -0.850 mm along the axis  → about 0.042 mm of blur
coma:             +0.0159 mm on the sensor   → about 0.016 mm of blur
chromatic:        -1.920 mm along the axis  → about 0.096 mm of blur
main problem: chromatic
blue light focuses closer to the lens than red, so white light smears into coloured fringes.
try: a glass with a bigger abbe number V (it spreads colours less).


'chromatic'

**a smaller opening doesn't fix colour.** at f/10 every blur shrinks (each colour's rays come in at a gentler slope), but chromatic aberration is still the main problem, at 0.096 mm. the blue and red foci are still 1.92 mm apart; closing the opening only makes the colour fringes thinner. the fix has to come from the glass.

In [7]:
fk51a = catalog[0]
n = fk51a["nd"]
R1 = 100 * (n - 1) / 0.5
flat = [(0, R1, n), (6, R2_for(R1, 6, 50, n), 1.0)]
diagnose(flat, z_stop = -15, glass = fk51a)

spherical:        -1.435 mm along the axis  → about 0.144 mm of blur
field curvature:  -0.850 mm along the axis  → about 0.085 mm of blur
coma:             +0.0020 mm on the sensor   → about 0.002 mm of blur
chromatic:        -0.582 mm along the axis  → about 0.058 mm of blur
main problem: spherical
rays through the edge of the lens focus closer than rays through the middle, even straight ahead.
try: a shape closer to best-form, a smaller opening, or a glass with a higher index.


'spherical'

**following the advice: a glass with a bigger V.** the same flat-field shape in N-FK51A (V = 84.5). chromatic blur drops from 0.192 to 0.058 mm (3.3x less) and is no longer the main problem. spherical aberration leads again (0.144 mm), because N-FK51A bends light weakly and needs stronger curves. every fix moves the problem somewhere else: diagnose, fix the biggest, diagnose again.

## what this shows

- colour is set by the glass (its abbe number V) and the focal length, not by the shape or the opening.
- picking a glass is a trade-off: high-index glasses cut spherical aberration but usually spread colour more. a few (N-SK16, N-LAK9) do both well.
- no single glass can make chromatic aberration zero: blue always bends more than red. real lenses fix this by gluing two glasses together, one that spreads colour a lot and one that spreads it little, so their colour errors cancel (an achromatic doublet). that's the natural next step for this project.

catalog values are SCHOTT's published nd and V.